# Обучение NNUE на Google Colab (CUDA)
## v5 1024s

### Перед запуском:
1. Runtime → Change runtime type → **T4 GPU**
2. Подключи Google Drive (кнопка слева → Mount Drive)
3. Запускай всё по порядку

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/drive/MyDrive/nnue_checkpoints
!mkdir -p /content/data

## 1. Клонируем bullet

In [ ]:
!git clone https://github.com/jw1912/bullet /content/bullet
%cd /content/bullet

## 2. Конфиг (без factoriser, чистая HalfKA)

In [ ]:
!mkdir -p /content/bullet/training/configs

In [ ]:
%%writefile /content/bullet/training/configs/v5_1024s_colab.rs
use bullet_lib::{
    game::{inputs::{ChessBucketsMirrored, get_num_buckets},
        outputs::MaterialCount},
    nn::optimiser::{AdamW, AdamWParams},
    trainer::{save::SavedFormat,
        schedule::{TrainingSchedule, TrainingSteps, lr, wdl},
        settings::LocalSettings},
    value::{ValueTrainerBuilder, loader::SfBinpackLoader},
};
use sfbinpack::chess::{piecetype::PieceType, r#move::MoveType};

fn main() {
    const FT_SIZE: usize = 1024;
    const NUM_OUTPUT_BUCKETS: usize = 8;

    let args: Vec<String> = std::env::args().collect();
    let dataset_dir = get_arg(&args, "--dataset-dir", "/content/data");
    let superbatches: usize = get_arg(&args, "--superbatches", "800").parse().unwrap();
    let wdl_proportion: f32 = get_arg(&args, "--wdl", "0.07").parse().unwrap();
    let initial_lr: f32 = get_arg(&args, "--lr", "0.001").parse().unwrap();
    let final_lr: f32 = get_arg(&args, "--final-lr",
        &format!("{}", initial_lr * 0.3f32.powi(5))).parse().unwrap();
    let save_rate: usize = get_arg(&args, "--save-rate", "25").parse().unwrap();
    let start_sb: usize = get_arg(&args, "--start-superbatch", "1").parse().unwrap();
    let load_ckpt: Option<String> = get_arg_opt(&args, "--load-checkpoint");
    let load_weights: Option<String> = get_arg_opt(&args, "--load-weights");

    #[rustfmt::skip]
    const BUCKET_LAYOUT: [usize; 32] = [
         0,  4,  8, 12,
         0,  4,  8, 12,
         1,  5,  9, 13,
         1,  5,  9, 13,
         2,  6, 10, 14,
         2,  6, 10, 14,
         3,  7, 11, 15,
         3,  7, 11, 15,
    ];
    const NUM_INPUT_BUCKETS: usize = get_num_buckets(&BUCKET_LAYOUT);

    let mut trainer = ValueTrainerBuilder::default()
        .dual_perspective()
        .optimiser(AdamW)
        .inputs(ChessBucketsMirrored::new(BUCKET_LAYOUT))
        .output_buckets(MaterialCount::<NUM_OUTPUT_BUCKETS>)
        .save_format(&[
            SavedFormat::id("l0w").round().quantise::<i16>(255),
            SavedFormat::id("l0b").round().quantise::<i16>(255),
            SavedFormat::id("l1w").round().quantise::<i16>(64),
            SavedFormat::id("l1b").round().quantise::<i32>(255 * 64),
        ])
        .loss_fn(|output, target| output.sigmoid().power_error(target, 2.5))
        .build(|builder, stm_inputs, ntm_inputs, output_buckets| {
            let l0 = builder.new_affine("l0", 768 * NUM_INPUT_BUCKETS, FT_SIZE);
            let l1 = builder.new_affine("l1", 2 * FT_SIZE, NUM_OUTPUT_BUCKETS);
            let stm = l0.forward(stm_inputs).screlu();
            let ntm = l0.forward(ntm_inputs).screlu();
            l1.forward(stm.concat(ntm)).select(output_buckets)
        });
    trainer.optimiser.set_params_for_weight("l0w",
        AdamWParams { max_weight: 0.99, min_weight: -0.99, ..Default::default() });

    if let Some(ckpt) = &load_ckpt {
        println!("Loading full checkpoint (weights + AdamW state) from: {}", ckpt);
        trainer.optimiser.load_from_checkpoint(ckpt)
            .expect("Failed to load checkpoint");
    }

    if let Some(wpath) = &load_weights {
        println!("Loading weights only from: {}", wpath);
        trainer.optimiser.load_weights_from_file(wpath)
            .expect("Failed to load weights");
    }

    let schedule = TrainingSchedule {
        net_id: "v5-1024s-colab".to_string(),
        eval_scale: 400.0,
        steps: TrainingSteps {
            batch_size: 16384,
            batches_per_superbatch: 6104,
            start_superbatch: start_sb,
            end_superbatch: superbatches,
        },
        wdl_scheduler: wdl::ConstantWDL { value: wdl_proportion },
        lr_scheduler: lr::CosineDecayLR { initial_lr, final_lr,
            final_superbatch: superbatches },
        save_rate,
    };
    let settings = LocalSettings {
        threads: 4, batch_queue_size: 64,
        output_directory: "/content/drive/MyDrive/nnue_checkpoints",
        test_set: None,
    };
    let filter = |e: &sfbinpack::TrainingDataEntry| {
        let stm = e.pos.side_to_move();
        e.ply >= 16 && !e.pos.is_checked(stm)
            && e.score.unsigned_abs() <= 10000
            && e.mv.mtype() == MoveType::Normal
            && e.pos.piece_at(e.mv.to()).piece_type() == PieceType::None
    };
    let data_files: Vec<String> = std::fs::read_dir(&dataset_dir)
        .unwrap().filter_map(|entry| {
            let p = entry.ok()?.path();
            if p.extension().map_or(false, |ext| ext == "binpack") {
                Some(p.to_string_lossy().to_string())
            } else { None }
        }).collect();
    assert!(!data_files.is_empty(), "No binpack in {}", dataset_dir);
    let refs: Vec<&str> = data_files.iter().map(|s| s.as_str()).collect();
    let dl = SfBinpackLoader::new_concat_multiple(&refs, 256, 4, filter);

    println!("=== v5 1024s SCReLU ===");
    println!("{}/{} SBs, WDL {}, LR {}→{}", superbatches,
        NUM_OUTPUT_BUCKETS, wdl_proportion, initial_lr, final_lr);
    trainer.run(&schedule, &settings, &dl);
}

fn get_arg(a: &[String], f: &str, d: &str) -> String {
    get_arg_opt(a, f).unwrap_or_else(|| d.to_string())
}

fn get_arg_opt(a: &[String], f: &str) -> Option<String> {
    a.iter().position(|s| s == f)
        .and_then(|i| a.get(i+1))
        .map(|s| s.to_string())
}

## 3. Регистрируем конфиг

In [ ]:
!echo '' >> /content/bullet/crates/bullet_lib/Cargo.toml
!echo '[[example]]' >> /content/bullet/crates/bullet_lib/Cargo.toml
!echo 'name = "v5_1024s_colab"' >> /content/bullet/crates/bullet_lib/Cargo.toml
!echo 'path = "../../training/configs/v5_1024s_colab.rs"' >> /content/bullet/crates/bullet_lib/Cargo.toml
!tail -5 /content/bullet/crates/bullet_lib/Cargo.toml

## 4. Качаем датасет (10.8 GB, ~10-30 мин)

In [ ]:
!wget -O /content/data/test80.binpack \
  "https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/test80-2022-08-aug-16tb7p.v6-dd.min.binpack?download=true"
!ls -lh /content/data/

## 5. Устанавливаем CUDA 12.5, Rust и собираем

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

In [ ]:
import os
os.environ['PATH'] = os.environ['PATH'] + ':/root/.cargo/bin'

In [ ]:
!wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
!sudo dpkg -i cuda-keyring_1.1-1_all.deb 2>/dev/null
!sudo apt-get update -qq 2>/dev/null
!sudo apt-get install -y -qq cuda-toolkit-12-5 2>&1 | tail -3

In [ ]:
import os
os.environ['CUDA_PATH'] = '/usr/local/cuda-12.5'
os.environ['PATH'] = '/usr/local/cuda-12.5/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-12.5/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

In [ ]:
!cd /content/bullet && cargo build --release --features cuda 2>&1

## 6. Запускаем обучение!
800 супербатчей ~24 часа. Чекпоинты каждые 25 SBs → Google Drive.

In [ ]:
import os, glob, subprocess, sys, re

checkpoint_dir = "/content/drive/MyDrive/nnue_checkpoints"

folders = sorted(glob.glob(f"{checkpoint_dir}/v5-1024s-colab-*"),
                 key=lambda f: int(re.search(r'v5-1024s-colab-(\d+)$', f).group(1))
                 if re.search(r'v5-1024s-colab-(\d+)$', f) else 0)

if folders:
    last = folders[-1]
    sb = int(re.search(r'v5-1024s-colab-(\d+)$', last).group(1))
    print(f"Продолжаем с SB {sb+1}")
    print(f"Чекпоинт: {last}/optimiser_state")
    resume = f"--start-superbatch {sb+1} --load-checkpoint {last}/optimiser_state"
else:
    print("Новый запуск с нуля")
    resume = ""

cmd = f"""cd /content/bullet && cargo run --release --features cuda --example v5_1024s_colab -- \
    --dataset-dir /content/data --superbatches 800 --save-rate 25 {resume}"""

print("🚀 Запускаем обучение...")
process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()
process.wait()


Estimated time to end of superbatch: 5.9s     superbatch 158 [96.5% (5888/6104 batches, 952603 pos/sec)]
Estimated time to end of superbatch: 3.7s     superbatch 158 [98.6% (6016/6104 batches, 953815 pos/sec)]
